# Notebook 59 — Differential Transport Law in k

Notebook 58 learned a sparse static symbolic meta-law:

\[
\beta = f(\text{metadata}, k)
\]

This notebook asks a sharper question:

\[
\frac{d\beta}{dk}=F(\beta,\text{metadata})
\]

Instead of predicting coefficients directly, we estimate how the coefficient vector **moves** as `k` changes.

## Main goals

1. Build coefficient trajectories across `k`.
2. Estimate finite-difference transport vectors \(d\beta/dk\).
3. Learn differential transport laws:
   - linear transport
   - ridge transport
   - sparse symbolic transport
   - no-explicit-k transport
4. Integrate from one `k` value to another.
5. Compare differential transport against static symbolic prediction.
6. Visualize transport vectors in coefficient PCA space.

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)

In [ ]:
# Data discovery and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append(
                                    {
                                        "system": system,
                                        "task": task,
                                        "forcing_mode": forcing_mode,
                                        "k": k,
                                        "flow_mode": flow_mode,
                                        "condition_coord": c,
                                        "residual": r,
                                        "predicted_flow": g + noise,
                                        "next_residual": next_r,
                                        "delta_condition": delta_c,
                                        "sample_id": sample_id,
                                        "path_id": path_id,
                                        "window_id": window_id,
                                    }
                                )
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

In [ ]:
# Loading + schema alignment

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        dc = np.gradient(df["condition_coord"].to_numpy())
        df["delta_condition"] = dc

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for k, v in defaults.items():
        if k not in df.columns:
            df[k] = v

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# Fixed governing template and per-regime coefficients

TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]
COEF_COLS = [f"coef_{t}" for t in TERM_NAMES]

def design_template(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return np.column_stack([
        np.ones_like(r),
        c,
        r,
        c**3,
        r**2,
        r * c**2,
    ])

def fit_template(sub, alpha=1e-6):
    X = design_template(sub)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    stats = {"n": len(sub), "rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(TERM_NAMES, beta):
        stats[f"coef_{name}"] = float(coef)
    return beta, pred, stats

def predict_with_beta(sub, beta):
    return design_template(sub) @ beta

def trajectory_gap(sub, beta_ref, beta_cmp, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i+1] - cgrid[i])
            x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
            g = float(np.clip(x @ beta, -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    errs = []
    for r0 in r0s:
        t_ref = integrate(beta_ref, r0)
        t_cmp = integrate(beta_cmp, r0)
        errs.append(math.sqrt(mean_squared_error(t_ref, t_cmp)))
    return float(np.mean(errs))

group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
regime_rows = []
regime_subsets = {}

for vals, sub in df.groupby(group_cols):
    if len(sub) < 30:
        continue
    beta, pred, stats = fit_template(sub.copy())
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
    row = {
        "regime_id": regime_id,
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
    }
    row.update(stats)
    regime_rows.append(row)
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(regime_rows).reset_index(drop=True)
display(coef_df.head())

## Build coefficient trajectories across k

Each trajectory is grouped by:

`system, task, forcing_mode, flow_mode`

and sorted by `k`.

In [ ]:
# Build coefficient trajectories and finite differences

traj_group_cols = ["system", "task", "forcing_mode", "flow_mode"]

transport_rows = []
trajectory_records = []

for vals, sub in coef_df.groupby(traj_group_cols):
    sub = sub.sort_values("k").reset_index(drop=True)
    if sub["k"].nunique() < 2:
        continue

    betas = sub[COEF_COLS].to_numpy(dtype=float)
    ks = sub["k"].to_numpy(dtype=float)

    trajectory_records.append({
        "meta": vals,
        "k": ks,
        "beta": betas,
        "rows": sub.copy(),
    })

    for i in range(len(sub) - 1):
        k0 = ks[i]
        k1 = ks[i + 1]
        dk = k1 - k0
        beta0 = betas[i]
        beta1 = betas[i + 1]
        dbeta_dk = (beta1 - beta0) / dk

        row = {
            "system": vals[0],
            "task": vals[1],
            "forcing_mode": vals[2],
            "flow_mode": vals[3],
            "k0": k0,
            "k1": k1,
            "dk": dk,
            "regime_id_0": sub.loc[i, "regime_id"],
            "regime_id_1": sub.loc[i + 1, "regime_id"],
        }

        for name, b in zip(COEF_COLS, beta0):
            row[f"{name}_at_k0"] = b
        for name, db in zip(COEF_COLS, dbeta_dk):
            row[f"d{name}_dk"] = db

        transport_rows.append(row)

transport_df = pd.DataFrame(transport_rows)
DCOEF_COLS = [f"d{name}_dk" for name in COEF_COLS]

display(transport_df.head())
print("Number of k-transport samples:", len(transport_df))

## Transport feature construction

Two forms are used:

1. `with_k`: current coefficient vector + metadata + `k0`
2. `no_k`: current coefficient vector + metadata only

The no-k experiment tests whether `k` is merely a coordinate along an intrinsic flow.

In [ ]:
# Transport features

BETA_AT_K0_COLS = [f"{c}_at_k0" for c in COEF_COLS]

def build_transport_features(df_in, include_k=True, include_interactions=True, columns=None):
    X = df_in[BETA_AT_K0_COLS].copy().astype(float)

    meta = pd.get_dummies(df_in[["system", "task", "forcing_mode", "flow_mode"]], drop_first=False)
    X = pd.concat([X, meta], axis=1)

    if include_k:
        X["k0"] = df_in["k0"].astype(float).values
        X["k0_2"] = df_in["k0"].astype(float).values ** 2

    if include_interactions:
        beta_norm = np.linalg.norm(df_in[BETA_AT_K0_COLS].to_numpy(dtype=float), axis=1)
        for fmode in sorted(df_in["forcing_mode"].astype(str).unique()):
            mask = (df_in["forcing_mode"].astype(str) == fmode).astype(float).values
            X[f"forcing_{fmode}_x_beta_norm"] = mask * beta_norm

        for flow in sorted(df_in["flow_mode"].astype(str).unique()):
            mask = (df_in["flow_mode"].astype(str) == flow).astype(float).values
            X[f"flow_{flow}_x_beta_norm"] = mask * beta_norm

    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)

    return X.astype(float)

X_transport = build_transport_features(transport_df, include_k=True)
display(X_transport.head())
print("Transport feature count:", X_transport.shape[1])

## Fit differential transport models

In [ ]:
# Multi-output differential models

def fit_predict_transport_models(train_df, test_df, include_k=True):
    X_train = build_transport_features(train_df, include_k=include_k)
    X_test = build_transport_features(test_df, include_k=include_k, columns=X_train.columns)

    Y_train = train_df[DCOEF_COLS].to_numpy(dtype=float)
    Y_test = test_df[DCOEF_COLS].to_numpy(dtype=float)

    lin = LinearRegression().fit(X_train, Y_train)
    rid = Ridge(alpha=1.0).fit(X_train, Y_train)

    preds = {
        "linear": lin.predict(X_test),
        "ridge": rid.predict(X_test),
    }

    Yhat_lasso = np.zeros_like(Y_test)
    lasso_models = []
    for j, target in enumerate(DCOEF_COLS):
        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(X_train)
        Xte_s = scaler.transform(X_test)

        model = LassoCV(cv=min(5, len(train_df)), random_state=42, max_iter=20000)
        model.fit(Xtr_s, Y_train[:, j])
        Yhat_lasso[:, j] = model.predict(Xte_s)
        lasso_models.append({"target": target, "model": model, "scaler": scaler, "columns": X_train.columns.tolist()})

    preds["symbolic"] = Yhat_lasso
    models = {"linear": lin, "ridge": rid, "symbolic": lasso_models}

    return preds, models, Y_test, X_train.columns.tolist()

def integrate_one_step(beta0, dbeta_dk, dk):
    return np.asarray(beta0, dtype=float) + float(dk) * np.asarray(dbeta_dk, dtype=float)

## Static symbolic baseline from Notebook 58

We compare differential transport against a static symbolic map:

\[
\beta = f(\text{metadata}, k)
\]

In [ ]:
# Static symbolic feature construction

def build_static_symbolic_features(df_in, columns=None):
    X = pd.get_dummies(df_in[["forcing_mode", "flow_mode", "system", "task"]], drop_first=False)

    X["k"] = df_in["k"].astype(float).values
    X["k2"] = df_in["k"].astype(float).values ** 2

    ff = df_in["forcing_mode"].astype(str) + "__x__" + df_in["flow_mode"].astype(str)
    X_ff = pd.get_dummies(ff, prefix="ff")

    sf = df_in["system"].astype(str) + "__x__" + df_in["forcing_mode"].astype(str)
    X_sf = pd.get_dummies(sf, prefix="sf")

    X_fk = pd.get_dummies(df_in["forcing_mode"].astype(str), prefix="fk").multiply(
        df_in["k"].astype(float).to_numpy().reshape(-1, 1)
    )

    X = pd.concat([X, X_ff, X_sf, X_fk], axis=1)

    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)

    return X.astype(float)

def fit_static_symbolic(train_coef_df, test_coef_df):
    X_train = build_static_symbolic_features(train_coef_df)
    X_test = build_static_symbolic_features(test_coef_df, columns=X_train.columns)

    Yhat = np.zeros((len(test_coef_df), len(COEF_COLS)), dtype=float)

    for j, coef in enumerate(COEF_COLS):
        y_train = train_coef_df[coef].to_numpy(dtype=float)
        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(X_train)
        Xte_s = scaler.transform(X_test)

        model = LassoCV(cv=min(5, len(train_coef_df)), random_state=42, max_iter=20000)
        model.fit(Xtr_s, y_train)
        Yhat[:, j] = model.predict(Xte_s)

    return Yhat

## One-step k transport evaluation

For each finite-difference sample, learn \(d\beta/dk\) on training samples,
predict the derivative for a held-out sample, and integrate:

\[
\hat\beta(k_1)=\beta(k_0)+(k_1-k_0)\hat F(\beta(k_0),metadata)
\]

In [ ]:
# Evaluation blocks for transport samples

transport_block_masks = {
    "target_k_high_7": transport_df["k1"].astype(float) == 7.0,
    "target_k_mid_5": transport_df["k1"].astype(float) == 5.0,
    "source_k_low_3": transport_df["k0"].astype(float) == 3.0,
}

rows = []

for block_name, test_mask in transport_block_masks.items():
    train_mask = ~test_mask

    train_t = transport_df.loc[train_mask].reset_index(drop=True)
    test_t = transport_df.loc[test_mask].reset_index(drop=True)

    if len(train_t) < 5 or len(test_t) == 0:
        continue

    preds_k, models_k, Ytrue_deriv, cols_k = fit_predict_transport_models(train_t, test_t, include_k=True)
    preds_nok, models_nok, _, cols_nok = fit_predict_transport_models(train_t, test_t, include_k=False)

    for i, row in test_t.iterrows():
        beta0 = row[BETA_AT_K0_COLS].to_numpy(dtype=float)
        beta1_true = coef_df.loc[coef_df["regime_id"] == row["regime_id_1"], COEF_COLS].to_numpy(dtype=float)[0]
        dk = row["dk"]
        rid1 = row["regime_id_1"]
        sub = regime_subsets[rid1]
        y_emp = sub["predicted_flow"].to_numpy(dtype=float)

        model_beta_preds = {}

        for name, yhat_deriv in preds_k.items():
            model_beta_preds[f"differential_{name}_with_k"] = integrate_one_step(beta0, yhat_deriv[i], dk)

        for name, yhat_deriv in preds_nok.items():
            model_beta_preds[f"differential_{name}_no_k"] = integrate_one_step(beta0, yhat_deriv[i], dk)

        target_regime = row["regime_id_1"]
        train_coef = coef_df[coef_df["regime_id"] != target_regime].reset_index(drop=True)
        test_coef = coef_df[coef_df["regime_id"] == target_regime].reset_index(drop=True)
        beta_static_symbolic = fit_static_symbolic(train_coef, test_coef)[0]
        model_beta_preds["static_symbolic"] = beta_static_symbolic

        for model_name, beta_hat in model_beta_preds.items():
            pred_field = predict_with_beta(sub, beta_hat)

            rows.append({
                "block": block_name,
                "source_regime": row["regime_id_0"],
                "target_regime": target_regime,
                "model": model_name,
                "coef_rmse": math.sqrt(mean_squared_error(beta1_true, beta_hat)),
                "field_rmse": math.sqrt(mean_squared_error(y_emp, pred_field)),
                "traj_rmse": trajectory_gap(sub, beta1_true, beta_hat),
            })

eval_df = pd.DataFrame(rows)
display(eval_df.head())

In [ ]:
# Summary by block/model

summary_df = eval_df.groupby(["block", "model"])[["coef_rmse", "field_rmse", "traj_rmse"]].mean().reset_index()
display(summary_df.sort_values(["block", "traj_rmse"]))

In [ ]:
# Model comparison plots

for metric in ["coef_rmse", "field_rmse", "traj_rmse"]:
    pivot = summary_df.pivot(index="block", columns="model", values=metric)
    ax = pivot.plot(kind="bar", figsize=(14, 5))
    ax.set_title(metric)
    ax.set_ylabel(metric)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## Transport vector field in coefficient PCA space

In [ ]:
# PCA space for coefficient manifold and transport vectors

coef_scaler = StandardScaler()
Ycoef_std = coef_scaler.fit_transform(coef_df[COEF_COLS].to_numpy(dtype=float))
pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(Ycoef_std)

coef_df["pc1"] = Z[:, 0]
coef_df["pc2"] = Z[:, 1]

transport_vec_rows = []
for _, row in transport_df.iterrows():
    beta0 = row[BETA_AT_K0_COLS].to_numpy(dtype=float)
    dbeta = row[DCOEF_COLS].to_numpy(dtype=float)
    beta1_est = beta0 + dbeta * row["dk"]

    z0 = pca.transform(coef_scaler.transform(beta0.reshape(1, -1)))[0]
    z1 = pca.transform(coef_scaler.transform(beta1_est.reshape(1, -1)))[0]
    dz = z1 - z0

    transport_vec_rows.append({
        "system": row["system"],
        "task": row["task"],
        "forcing_mode": row["forcing_mode"],
        "flow_mode": row["flow_mode"],
        "k0": row["k0"],
        "k1": row["k1"],
        "z0_1": z0[0],
        "z0_2": z0[1],
        "dz1": dz[0],
        "dz2": dz[1],
        "norm_dz": float(np.linalg.norm(dz)),
    })

transport_vec_df = pd.DataFrame(transport_vec_rows)
display(transport_vec_df.head())

In [ ]:
# Plot true k-transport arrows in coefficient PCA space

plt.figure(figsize=(9, 7))
for fmode in sorted(coef_df["forcing_mode"].astype(str).unique()):
    sub = coef_df[coef_df["forcing_mode"].astype(str) == fmode]
    plt.scatter(sub["pc1"], sub["pc2"], label=fmode, alpha=0.5)

for _, row in transport_vec_df.iterrows():
    plt.arrow(
        row["z0_1"], row["z0_2"],
        row["dz1"], row["dz2"],
        head_width=0.04,
        length_includes_head=True,
        alpha=0.35,
    )

plt.title("Coefficient manifold transport arrows as k changes")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.tight_layout()
plt.show()

## Symbolic differential equations

Print sparse equations for \(d\beta/dk\) using all transport samples.

In [ ]:
# Fit full symbolic differential equations for reporting

X_full = build_transport_features(transport_df, include_k=False)
Y_full = transport_df[DCOEF_COLS].to_numpy(dtype=float)

diff_equation_rows = []

for j, target in enumerate(DCOEF_COLS):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_full)
    model = LassoCV(cv=min(5, len(transport_df)), random_state=42, max_iter=20000)
    model.fit(Xs, Y_full[:, j])

    terms = []
    for feat, coef_val in zip(X_full.columns, model.coef_):
        if abs(coef_val) > 1e-3:
            terms.append((feat, coef_val))

    parts = [f"{target} = {model.intercept_:.5f}"]
    for feat, coef_val in terms:
        sign = "+" if coef_val >= 0 else "-"
        parts.append(f"{sign} {abs(coef_val):.5f}·{feat}")

    eq = " ".join(parts)
    diff_equation_rows.append({
        "target": target,
        "equation": eq,
        "n_terms": len(terms),
    })

    print(eq)
    print()

diff_eq_df = pd.DataFrame(diff_equation_rows)
display(diff_eq_df)

## Representative transport case

In [ ]:
# Pick a case where differential symbolic no-k beats static symbolic on trajectory RMSE

case_df = eval_df.pivot_table(
    index=["block", "source_regime", "target_regime"],
    columns="model",
    values="traj_rmse",
    aggfunc="mean",
).reset_index()

if "differential_symbolic_no_k" in case_df.columns and "static_symbolic" in case_df.columns:
    case_df["gain"] = case_df["static_symbolic"] - case_df["differential_symbolic_no_k"]
    rep = case_df.sort_values("gain", ascending=False).iloc[0]
else:
    rep = case_df.iloc[0]

source_regime = rep["source_regime"]
target_regime = rep["target_regime"]

beta0 = coef_df.loc[coef_df["regime_id"] == source_regime, COEF_COLS].to_numpy(dtype=float)[0]
beta_true = coef_df.loc[coef_df["regime_id"] == target_regime, COEF_COLS].to_numpy(dtype=float)[0]
sub_target = regime_subsets[target_regime]

sample = transport_df[(transport_df["regime_id_0"] == source_regime) & (transport_df["regime_id_1"] == target_regime)].iloc[0:1].reset_index(drop=True)

train_t = transport_df[~(
    (transport_df["regime_id_0"] == source_regime) & (transport_df["regime_id_1"] == target_regime)
)].reset_index(drop=True)

preds_nok, _, _, _ = fit_predict_transport_models(train_t, sample, include_k=False)
beta_diff_symbolic = integrate_one_step(beta0, preds_nok["symbolic"][0], sample.loc[0, "dk"])

train_coef = coef_df[coef_df["regime_id"] != target_regime].reset_index(drop=True)
test_coef = coef_df[coef_df["regime_id"] == target_regime].reset_index(drop=True)
beta_static_symbolic = fit_static_symbolic(train_coef, test_coef)[0]

coef_compare = pd.DataFrame({
    "term": COEF_COLS,
    "source_beta": beta0,
    "true_target": beta_true,
    "diff_symbolic_no_k": beta_diff_symbolic,
    "static_symbolic": beta_static_symbolic,
})
display(coef_compare)

In [ ]:
# Representative trajectory comparison

cmin, cmax = sub_target["condition_coord"].min(), sub_target["condition_coord"].max()
rmin, rmax = sub_target["residual"].min(), sub_target["residual"].max()
cgrid = np.linspace(cmin, cmax, 45)
r0s = np.linspace(np.quantile(sub_target["residual"], 0.1), np.quantile(sub_target["residual"], 0.9), 8)
flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub_target["predicted_flow"]), 0.995))

def integrate_beta(beta, r0):
    r = float(np.clip(r0, rmin, rmax))
    traj = [r]
    for i in range(len(cgrid) - 1):
        c = cgrid[i]
        dc = float(cgrid[i+1] - cgrid[i])
        x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
        g = float(np.clip(x @ beta, -flow_cap, flow_cap))
        r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
        traj.append(r)
    return np.array(traj)

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
titles = ["Source", "True target", "Differential symbolic", "Static symbolic"]
betas = [beta0, beta_true, beta_diff_symbolic, beta_static_symbolic]

for ax, title, beta in zip(axes, titles, betas):
    for r0 in r0s:
        ax.plot(cgrid, integrate_beta(beta, r0))
    ax.set_title(title)
    ax.set_xlabel("condition coordinate c")
axes[0].set_ylabel("residual")
fig.suptitle(f"Transport: {source_regime} → {target_regime}", y=1.03)
plt.tight_layout()
plt.show()

## Final verdicts

In [ ]:
# Best model by trajectory RMSE per block

verdict_rows = []
for block, sub in summary_df.groupby("block"):
    best = sub.sort_values("traj_rmse").iloc[0]
    verdict_rows.append({
        "block": block,
        "best_model": best["model"],
        "best_traj_rmse": best["traj_rmse"],
        "best_field_rmse": best["field_rmse"],
        "best_coef_rmse": best["coef_rmse"],
    })

verdict_df = pd.DataFrame(verdict_rows)
display(verdict_df)

## Working conclusion

Notebook 59 turns the static coefficient meta-law into a differential transport law in `k`.

A strong result is:
- \(d\beta/dk\) can be predicted from the current coefficient state and metadata
- no-explicit-k transport remains competitive, suggesting `k` acts like a coordinate along a coefficient flow
- differential integration can match or improve static symbolic extrapolation in selected k-transport cases

If this pattern holds on your real exports, the next notebook should be:

**Notebook 60 — continuous ODE coefficient transport and multi-step integration**